## Step 1: Regression Dataset Class Definition (`RegressionBauteil`)

In [ ]:
import numpy as np
from functools import cached_property
import os
import sys
from pathlib import Path

# --- 1. Resolve Project Root ('MiniPresse') & Output Directory ---
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "DataAcquisition.py").exists() and (PROJECT_ROOT.parent / "DataAcquisition.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "models" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import DataAcquisition as da


# --- 2. Specimen Wrapper Class ---
class RegressionBauteil(da.Bauteil):
    """
    Extends da.Bauteil by delegating file I/O and metadata access to base properties
    and resampling Force-Displacement F(s) curves into fixed-length segments.
    """
    def __init__(self, bauteil_id: int, n_load: int = 200, n_unload: int = 100, f_thresh: float = 10.0):
        super().__init__(bauteil_id)
        self.id = bauteil_id
        self.n_load = n_load
        self.n_unload = n_unload
        self.f_thresh = f_thresh

    @cached_property
    def angle(self) -> float | None:
        val = self.Qualitaetsdaten
        if val is None or np.isnan(val):
            return None
        return float(val)

    @cached_property
    def material_name(self) -> str:
        if self.Material is None:
            return "unknown"
        return getattr(self.Material, "name", getattr(self.Material, "material", str(self.Material)))

    @cached_property
    def has_vorbiegen(self) -> bool:
        meta_text = " ".join([
            str(self.error or ""),
            str(self.actions or ""),
            str(self.observations or ""),
            self.material_name,
            str(self.Machine or "")
        ]).lower()
        return "vorbieg" in meta_text or "vorbeug" in meta_text or "vorgebog" in meta_text

    @cached_property
    def is_valid_for_regression(self) -> bool:
        err = str(self.error or "").strip().lower()
        if err not in ("", "none", "false", "kein_fehler", "kein fehler", "vorgebogenes_material"):
            return False
        if self.angle is None:
            return False
        return self.kraft_weg_data is not None

    @cached_property
    def kraft_weg_data(self) -> dict | None:
        try:
            df = self.Prozessdaten
        except Exception:
            return None

        if df is None or df.empty:
            return None

        force = df["Force"].values
        pos = df["Position"].values

        # Filter active contact region (remove empty approach/retract travel)
        contact_mask = force > self.f_thresh
        if np.sum(contact_mask) < 20:
            return None

        f_active = force[contact_mask]
        p_active = pos[contact_mask]

        # Find bottom dead center (BDC / UT)
        idx_ut = np.argmin(p_active)
        if idx_ut < 10 or (len(p_active) - idx_ut) < 5:
            return None

        # Loading phase (Position decreasing -> reverse array for np.interp)
        p_load = p_active[:idx_ut + 1][::-1]
        f_load = f_active[:idx_ut + 1][::-1]
        p_load_u, idx_u = np.unique(p_load, return_index=True)
        f_load_u = f_load[idx_u]

        grid_load = np.linspace(p_load_u.min(), p_load_u.max(), self.n_load)
        f_res_load = np.interp(grid_load, p_load_u, f_load_u)[::-1]

        # Unloading / springback phase (Position increasing from BDC)
        p_unload = p_active[idx_ut:]
        f_unload = f_active[idx_ut:]
        p_unload_u, idx_uu = np.unique(p_unload, return_index=True)
        f_unload_u = f_unload[idx_uu]

        grid_unload = np.linspace(p_unload_u.min(), p_unload_u.max(), self.n_unload)
        f_res_unload = np.interp(grid_unload, p_unload_u, f_unload_u)

        # Physical scalars preserving absolute stroke, peak force, and sheet thickness
        stroke_load = float(p_load_u.max() - p_load_u.min())
        stroke_unload = float(p_unload_u.max() - p_unload_u.min())
        f_max = float(f_active.max())
        raw_thick = getattr(self.Material, "thickness", getattr(self.Material, "s", 0.0))
        thickness = float(raw_thick) if raw_thick is not None else 0.0

        return {
            "curve": np.concatenate([f_res_load, f_res_unload]).astype(np.float32),
            "scalars": np.array([stroke_load, stroke_unload, f_max, thickness], dtype=np.float32),
        }

    @property
    def curve(self) -> np.ndarray:
        return self.kraft_weg_data["curve"]

    @property
    def scalars(self) -> np.ndarray:
        return self.kraft_weg_data["scalars"]

    @classmethod
    def load_subset(
        cls,
        min_id: int = 21,
        max_id: int | None = None,
        material_filter: str | None = None,
        vorbiegen: bool | None = None,
        n_load: int = 200,
        n_unload: int = 100
    ):
        selected = []
        for b_id in sorted(da.get_ids()):
            if b_id < min_id:
                continue
            if max_id is not None and b_id > max_id:
                continue

            rb = cls(b_id, n_load=n_load, n_unload=n_unload)
            if not rb.is_valid_for_regression:
                continue
            if material_filter is not None and material_filter.lower() not in rb.material_name.lower():
                continue
            if vorbiegen is not None and rb.has_vorbiegen != vorbiegen:
                continue

            selected.append(rb)

        X_curves = np.stack([b.curve for b in selected])
        X_scalars = np.stack([b.scalars for b in selected])
        y_angles = np.array([b.angle for b in selected], dtype=np.float32)

        return selected, X_curves, X_scalars, y_angles


# --- 3. Load Non-Vorbiegen Subset & Preload Contiguous Arrays into RAM ---
bauteile, X_curves, X_scalars, y_angles = RegressionBauteil.load_subset(
    min_id=21,
    vorbiegen=False
)

## Step 2: Data Preloading & Model Definition

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


# --- Block 1: RAM Preloading & Model Architecture ---
X_curves_ram = np.ascontiguousarray(X_curves, dtype=np.float32)
X_scalars_ram = np.ascontiguousarray(X_scalars, dtype=np.float32)
y_angles_ram = np.ascontiguousarray(y_angles, dtype=np.float32)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KFOLD_SPLITS = list(KFold(n_splits=5, shuffle=True, random_state=42).split(y_angles_ram))

print(
    f"Loaded {len(y_angles_ram)} non-Vorbiegen specimens | "
    f"Angle range: {y_angles_ram.min():.2f}° – {y_angles_ram.max():.2f}° | "
    f"Device: {DEVICE}"
)


# --- 4. Hybrid 1D-CNN Architecture (Curve + Physical Scalars) ---
class BendingCNN1D(nn.Module):
    """Hybrid 1D-CNN fusing F(s) curve representations with physical process scalars."""
    def __init__(self, seq_len: int = 300, n_scalars: int = 4):
        super().__init__()
        self.conv_encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(8)
        )
        self.regressor = nn.Sequential(
            nn.Linear(64 * 8 + n_scalars, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 1)
        )

    def forward(self, curve: torch.Tensor, scalars: torch.Tensor) -> torch.Tensor:
        conv_feat = self.conv_encoder(curve).flatten(start_dim=1)
        fused = torch.cat([conv_feat, scalars], dim=1)
        return self.regressor(fused).squeeze(dim=-1)

## Step 3: Cross-Validation Model Training

In [ ]:
# --- Block 2: Training Pipeline Across All Experiments (Non-Vorbiegen) ---
def train_cnn_model(
    c_tr: np.ndarray,
    s_tr: np.ndarray,
    y_tr: np.ndarray,
    epochs: int = 160,
    batch_size: int = 16,
    lr: float = 1e-3,
    seed: int = 42
) -> BendingCNN1D:
    """Trains a single 1D-CNN model on the preloaded training fold."""
    torch.manual_seed(seed)

    t_c_tr = torch.tensor(c_tr[:, None, :], dtype=torch.float32, device=DEVICE)
    t_s_tr = torch.tensor(s_tr, dtype=torch.float32, device=DEVICE)
    t_y_tr = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)

    loader = DataLoader(
        TensorDataset(t_c_tr, t_s_tr, t_y_tr),
        batch_size=batch_size,
        shuffle=True
    )

    model = BendingCNN1D(seq_len=c_tr.shape[1], n_scalars=s_tr.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.HuberLoss(delta=1.0)

    model.train()
    for _ in range(epochs):
        for bc, bs, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bc, bs), by)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    return model


def train_experiment_folds(
    name: str,
    curves: np.ndarray | None,
    scalars: np.ndarray | None,
    y: np.ndarray,
    model_type: str = "ridge",
    train_frac: float = 1.0
) -> dict:
    """Fits scalers and models for every CV fold and stores validation artifacts."""
    rng = np.random.default_rng(42)
    fold_records = []

    for fold, (tr_idx, va_idx) in enumerate(KFOLD_SPLITS, start=1):
        if train_frac < 1.0:
            k = max(8, int(len(tr_idx) * train_frac))
            tr_idx = rng.choice(tr_idx, size=k, replace=False)

        y_tr, y_va = y[tr_idx], y[va_idx]
        feats_tr, feats_va = [], []
        c_tr = c_va = s_tr = s_va = None

        # Fit scalers strictly on the training fold to prevent data leakage
        if curves is not None:
            sc_c = StandardScaler()
            c_tr = sc_c.fit_transform(curves[tr_idx])
            c_va = sc_c.transform(curves[va_idx])
            feats_tr.append(c_tr)
            feats_va.append(c_va)

        if scalars is not None:
            sc_s = StandardScaler()
            s_tr = sc_s.fit_transform(scalars[tr_idx])
            s_va = sc_s.transform(scalars[va_idx])
            feats_tr.append(s_tr)
            feats_va.append(s_va)

        X_tr = np.hstack(feats_tr)
        X_va = np.hstack(feats_va)

        if model_type == "ridge":
            model = Ridge(alpha=10.0).fit(X_tr, y_tr)
        elif model_type == "rf":
            model = RandomForestRegressor(
                n_estimators=300, max_depth=12, n_jobs=-1, random_state=42
            ).fit(X_tr, y_tr)
        elif model_type == "cnn":
            model = train_cnn_model(c_tr, s_tr, y_tr, seed=42 + fold)
        else:
            raise ValueError(f"Unsupported model_type: {model_type}")

        fold_records.append({
            "fold": fold,
            "n_train": len(tr_idx),
            "model": model,
            "X_va": X_va,
            "c_va": c_va,
            "s_va": s_va,
            "y_va": y_va,
        })

    return {"name": name, "model_type": model_type, "folds": fold_records}


experiment_configs = [
    # Feature & curve-phase ablation (Ridge & Random Forest)
    ("1. Ridge  | Only 4 scalars", None, X_scalars_ram, "ridge", 1.0),
    ("2. Ridge  | Only F(s) curve (300 pts)", X_curves_ram, None, "ridge", 1.0),
    ("3. Ridge  | Loading phase (200 pts) + scalars", X_curves_ram[:, :200], X_scalars_ram, "ridge", 1.0),
    ("4. Ridge  | Unloading phase (100 pts) + scalars", X_curves_ram[:, 200:], X_scalars_ram, "ridge", 1.0),
    ("5. Ridge  | Full F(s) curve (300 pts) + scalars", X_curves_ram, X_scalars_ram, "ridge", 1.0),

    ("6. RF     | Only 4 scalars", None, X_scalars_ram, "rf", 1.0),
    ("7. RF     | Full F(s) curve (300 pts) + scalars", X_curves_ram, X_scalars_ram, "rf", 1.0),

    # Hybrid 1D-CNN (Curve phases & full multimodal fusion)
    ("8. 1D-CNN | Loading phase (200 pts) + scalars", X_curves_ram[:, :200], X_scalars_ram, "cnn", 1.0),
    ("9. 1D-CNN | Unloading phase (100 pts) + scalars", X_curves_ram[:, 200:], X_scalars_ram, "cnn", 1.0),
    ("10. 1D-CNN| Full F(s) curve (300 pts) + scalars", X_curves_ram, X_scalars_ram, "cnn", 1.0),

    # Sample efficiency (Learning curve across specimen counts)
    ("11. 1D-CNN (25% train IDs)", X_curves_ram, X_scalars_ram, "cnn", 0.25),
    ("12. 1D-CNN (50% train IDs)", X_curves_ram, X_scalars_ram, "cnn", 0.50),
    ("13. 1D-CNN (75% train IDs)", X_curves_ram, X_scalars_ram, "cnn", 0.75),
]

trained_experiments = [
    train_experiment_folds(n, c, s, y_angles_ram, m, f)
    for n, c, s, m, f in experiment_configs
]
print(f"Finished training {len(trained_experiments)} non-Vorbiegen experiments ({len(trained_experiments) * 5} models).")

## Step 4: Validation Evaluation & CSV Export

In [ ]:
# --- Block 3: Validation Evaluation & CSV Export (Non-Vorbiegen) ---
eval_rows = []

for exp in trained_experiments:
    fold_maes, fold_rmses, fold_r2s = [], [], []
    n_train_used = 0

    for rec in exp["folds"]:
        n_train_used = rec["n_train"]
        model = rec["model"]
        y_va = rec["y_va"]

        if exp["model_type"] == "cnn":
            with torch.no_grad():
                t_c_va = torch.tensor(rec["c_va"][:, None, :], dtype=torch.float32, device=DEVICE)
                t_s_va = torch.tensor(rec["s_va"], dtype=torch.float32, device=DEVICE)
                preds = model(t_c_va, t_s_va).cpu().numpy()
        else:
            preds = model.predict(rec["X_va"])

        fold_maes.append(mean_absolute_error(y_va, preds))
        fold_rmses.append(root_mean_squared_error(y_va, preds))
        fold_r2s.append(r2_score(y_va, preds))

    eval_rows.append({
        "Experiment": exp["name"],
        "Train_IDs": n_train_used,
        "MAE_Mean": round(float(np.mean(fold_maes)), 4),
        "MAE_STD": round(float(np.std(fold_maes)), 4),
        "MAE_Summary": f"{np.mean(fold_maes):.3f} ± {np.std(fold_maes):.3f}",
        "RMSE_Mean": round(float(np.mean(fold_rmses)), 4),
        "RMSE_STD": round(float(np.std(fold_rmses)), 4),
        "RMSE_Summary": f"{np.mean(fold_rmses):.3f} ± {np.std(fold_rmses):.3f}",
        "R2_Mean": round(float(np.mean(fold_r2s)), 4),
        "R2_STD": round(float(np.std(fold_r2s)), 4),
        "R2_Summary": f"{np.mean(fold_r2s):.3f} ± {np.std(fold_r2s):.3f}",
    })

df_stats_no_vorbiegen = pd.DataFrame(eval_rows)

output_csv = RESULTS_DIR / "benchmark_results_no_vorbiegen.csv"
df_stats_no_vorbiegen.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Saved non-Vorbiegen statistical benchmark table to '{output_csv}'")
display(df_stats_no_vorbiegen[["Experiment", "Train_IDs", "MAE_Summary", "RMSE_Summary", "R2_Summary"]])

## Step 5: Single-Specimen Inference & Error Inspection

In [ ]:
# --- Block 4: Out-of-Fold Single-Specimen Inference ---
def predict_single_id(target_id: int, exp_index: int = 6):
    """
    Predicts the bending angle for a specific non-Vorbiegen specimen ID using the
    out-of-fold model where target_id was in the validation set (zero data leakage).
    """
    id_list = [b.id for b in bauteile]
    if target_id not in id_list:
        print(f"ID {target_id} is not in the non-Vorbiegen dataset (filtered out or Vorbiegen).")
        return

    sample_idx = id_list.index(target_id)
    bauteil = bauteile[sample_idx]
    true_angle = y_angles_ram[sample_idx]

    target_fold = None
    for fold_idx, (_, va_idx) in enumerate(KFOLD_SPLITS):
        if sample_idx in va_idx:
            target_fold = fold_idx
            va_pos = int(np.where(va_idx == sample_idx)[0][0])
            break

    exp = trained_experiments[exp_index]
    rec = exp["folds"][target_fold]
    model = rec["model"]

    if exp["model_type"] == "cnn":
        with torch.no_grad():
            t_c = torch.tensor(rec["c_va"][va_pos:va_pos+1, None, :], dtype=torch.float32, device=DEVICE)
            t_s = torch.tensor(rec["s_va"][va_pos:va_pos+1], dtype=torch.float32, device=DEVICE)
            pred_angle = float(model(t_c, t_s).cpu().numpy()[0])
    else:
        pred_angle = float(model.predict(rec["X_va"][va_pos:va_pos+1])[0])

    abs_error = abs(pred_angle - true_angle)

    print(f"=== Prediction Report for ID {target_id} (Fold {target_fold + 1}) ===")
    print(f"Model:          {exp['name']}")
    print(f"Material:       {bauteil.material_name} | Vorbiegen: {bauteil.has_vorbiegen}")
    print(f"Scalars:        stroke_load={bauteil.scalars[0]:.2f} mm, stroke_unload={bauteil.scalars[1]:.2f} mm, F_max={bauteil.scalars[2]:.1f} N, s={bauteil.scalars[3]:.2f} mm")
    print(f"Real Angle:     {true_angle:.3f}°")
    print(f"Predicted:      {pred_angle:.3f}°")
    print(f"Absolute Error: {abs_error:.3f}°")


# Test with a valid non-Vorbiegen ID (e.g., first loaded ID)
predict_single_id(bauteile[0].id)